In [4]:
import sys
!{sys.executable} -m pip install scikit-learn catboost

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------  7.9/8.0 MB 62.7 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 53.5 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2

In [7]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    classification_report,
    confusion_matrix
)
cbb = pd.read_csv("cbb2_ranked.csv")

In [9]:
cbb = pd.read_csv("cbb2_ranked.csv")

In [10]:
cbb = cbb.dropna(subset=["POSTSEASON"]).copy()
cbb = cbb[cbb["POSTSEASON"].astype(str).str.strip().ne("")].copy()

In [17]:
cbb["POSTSEASON"] = cbb["POSTSEASON"].replace({"R68": "R64"})
round_ordered = ["R64", "R32", "S16", "E8", "F4", "2ND", "Champions"]
round_i = {r: i for i, r in enumerate(round_ordered)}
idx_to_round = {i: r for r, i in round_i.items()}

cbb["round_idx"] = cbb["POSTSEASON"].map(round_i)
cbb = cbb.dropna(subset=["round_idx"]).copy()
cbb["round_idx"] = cbb["round_idx"].astype(int)
feature_cols = [
    "W",
    "ADJOE",
    "ADJDE",
    "BARTHAG",
    "EFG_O",
    "EFG_D",
    "TOR",
    "TORD",
    "FTR",
    "3P_O",
    "3P_D",
    "ADJ_T",
    "WAB",
    "SEED",
    "RK"
]


In [18]:
train = cbb[cbb["YEAR"] <= 2023].copy()
valid = cbb[cbb["YEAR"] == 2024].copy()

X_train = train[feature_cols]
y_train = train["round_idx"]

X_valid = valid[feature_cols]
y_valid = valid["round_idx"]

print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)
print("Train class counts:")
print(y_train.value_counts().sort_index())
print("Valid class counts:")
print(y_valid.value_counts().sort_index())


Train shape: (680, 15)
Valid shape: (68, 15)
Train class counts:
round_idx
0    360
1    160
2     80
3     40
4     20
5     10
6     10
Name: count, dtype: int64
Valid class counts:
round_idx
0    36
1    16
2     8
3     4
4     2
5     1
6     1
Name: count, dtype: int64


In [19]:
model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 1.8641673	test: 1.8753561	best: 1.8753561 (0)	total: 157ms	remaining: 1m 18s
100:	learn: 0.8324790	test: 1.0931838	best: 1.0931838 (100)	total: 527ms	remaining: 2.08s
200:	learn: 0.6420626	test: 1.0675009	best: 1.0675009 (200)	total: 911ms	remaining: 1.35s
300:	learn: 0.5116092	test: 1.0570280	best: 1.0550949 (294)	total: 1.28s	remaining: 846ms
400:	learn: 0.4081301	test: 1.0594459	best: 1.0550949 (294)	total: 1.74s	remaining: 431ms
499:	learn: 0.3293017	test: 1.0647058	best: 1.0550949 (294)	total: 2.12s	remaining: 0us

bestTest = 1.055094934
bestIteration = 294

Shrink model to first 295 iterations.


CatBoostClassifier(depth=6, eval_metric='MultiClass', iterations=500, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

In [20]:
pred_valid = model.predict(X_valid)
pred_valid = pred_valid.astype(int).ravel()

proba_valid = model.predict_proba(X_valid)

In [27]:
acc = accuracy_score(y_valid, pred_valid)
ll = log_loss(y_valid, proba_valid)

print("Accuracy:", acc)
print("LogLoss:", ll)
print("\nClassification report:\n")
print(classification_report(y_valid, pred_valid, target_names=round_ordered))

Accuracy: 0.5588235294117647
LogLoss: 1.055094933737938

Classification report:

              precision    recall  f1-score   support

         R64       0.71      0.75      0.73        36
         R32       0.32      0.38      0.34        16
         S16       0.56      0.62      0.59         8
          E8       0.00      0.00      0.00         4
          F4       0.00      0.00      0.00         2
         2ND       0.00      0.00      0.00         1
   Champions       0.00      0.00      0.00         1

    accuracy                           0.56        68
   macro avg       0.23      0.25      0.24        68
weighted avg       0.52      0.56      0.54        68



c:\Users\bfunk\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\bfunk\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\bfunk\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
cm = confusion_matrix(y_valid, pred_valid)
cm_df = pd.DataFrame(cm, index=round_ordered, columns=round_ordered)
print(cm_df)


           R64  R32  S16  E8  F4  2ND  Champions
R64         27    7    2   0   0    0          0
R32          9    6    1   0   0    0          0
S16          0    2    5   1   0    0          0
E8           0    3    1   0   0    0          0
F4           2    0    0   0   0    0          0
2ND          0    1    0   0   0    0          0
Champions    0    0    0   0   1    0          0


In [ ]:
feat_imp = pd.Series(
    model.get_feature_importance(),
    index=feature_cols
).sort_values(ascending=False)

print(feat_imp)

W          10.676568
BARTHAG     9.382615
FTR         8.103337
ADJOE       7.079417
ADJDE       6.877715
TOR         6.662627
ADJ_T       6.518127
3P_O        6.218437
EFG_O       6.164882
3P_D        6.113863
WAB         5.852292
RK          5.675573
TORD        5.594378
EFG_D       4.765576
SEED        4.314591
dtype: float64


In [ ]:
results_2024 = valid[["TEAM", "YEAR", "POSTSEASON", "round_idx"]].copy()
results_2024["pred_round_idx"] = pred_valid
results_2024["pred_round"] = results_2024["pred_round_idx"].map(idx_to_round)
proba_df = pd.DataFrame(
    proba_valid,
    columns=[f"proba_{idx_to_round[i]}" for i in range(len(round_ordered))]
)

results_2024 = pd.concat(
    [results_2024.reset_index(drop=True), proba_df.reset_index(drop=True)],
    axis=1
)

print(results_2024.head(20))

              TEAM  YEAR POSTSEASON  round_idx  pred_round_idx pred_round  \
0          Houston  2024        S16          2               2        S16   
1      Connecticut  2024  Champions          6               4         F4   
2           Purdue  2024        2ND          5               1        R32   
3   North Carolina  2024        S16          2               2        S16   
4         Iowa St.  2024        S16          2               3         E8   
5          Arizona  2024        S16          2               2        S16   
6        Tennessee  2024         E8          3               1        R32   
7        Marquette  2024        S16          2               1        R32   
8        Creighton  2024        S16          2               2        S16   
9         Illinois  2024         E8          3               1        R32   
10          Baylor  2024        R32          1               2        S16   
11        Kentucky  2024        R64          0               2        S16   

In [32]:
proba = model.predict_proba(X_valid)

round_weights = np.array([0, 1, 2, 4, 8, 12, 20])  # example weights
results_2024["power_score"] = (proba * round_weights).sum(axis=1)

mn = results_2024["power_score"].min()
mx = results_2024["power_score"].max()

print(results_2024.sort_values("power_score_100", ascending=False).head(25))

                TEAM POSTSEASON  power_score  power_score_100
3877     Connecticut  Champions     7.264255       100.000000
3876         Houston        S16     4.892948        82.564984
3878          Purdue        2ND     4.886485        78.709554
3880        Iowa St.        S16     3.243155        68.796519
3888          Auburn        R64     3.309022        68.320394
3881         Arizona        S16     2.758241        58.045275
3882       Tennessee         E8     2.237847        51.624976
3883       Marquette        S16     2.474779        50.903588
3879  North Carolina        S16     2.185874        50.624897
3889            Duke         E8     1.965739        49.541362
3884       Creighton        S16     2.363790        49.481275
3885        Illinois         E8     2.029477        45.708202
3893         Gonzaga        S16     1.816840        43.592292
3892    Saint Mary's        R64     1.862811        42.305464
3886          Baylor        R32     1.647107        42.201100
3891    

In [36]:
bracket_2024 = {
    "East": [
        ("Connecticut", "Stetson"),
        ("Florida Atlantic", "Northwestern"),
        ("San Diego St.", "UAB"),
        ("Auburn", "Yale"),
        ("BYU", "Duquesne"),
        ("Illinois", "Morehead St."),
        ("Washington St.", "Drake"),
        ("Iowa St.", "South Dakota St."),
    ],
    "West": [
        ("North Carolina", "Wagner"),
        ("Mississippi St.", "Michigan St."),
        ("Saint Mary's", "Grand Canyon"),
        ("Alabama", "College of Charleston"),
        ("Clemson", "New Mexico"),
        ("Baylor", "Colgate"),
        ("Dayton", "Nevada"),
        ("Arizona", "Long Beach St."),
    ],
    "South": [
        ("Houston", "Longwood"),
        ("Nebraska", "Texas A&M"),
        ("Wisconsin", "James Madison"),
        ("Duke", "Vermont"),
        ("Texas Tech", "North Carolina St."),
        ("Kentucky", "Oakland"),
        ("Florida", "Colorado"),
        ("Marquette", "Western Kentucky"),
    ],
    "Midwest": [
        ("Purdue", "Grambling St."),
        ("Utah St.", "TCU"),
        ("Gonzaga", "McNeese St."),
        ("Kansas", "Samford"),
        ("South Carolina", "Oregon"),
        ("Creighton", "Akron"),
        ("Texas", "Colorado St."),
        ("Tennessee", "Saint Peter's"),
    ]
}

# make lookup from team -> power score
power_lookup = dict(zip(results_2024["TEAM"], results_2024["power_score"]))

def get_power(team):
    if team not in power_lookup:
        raise KeyError(f"{team} not found in results_2024['TEAM']")
    return power_lookup[team]

def win_prob(team_a, team_b, temperature=0.5):
    a = get_power(team_a)
    b = get_power(team_b)
    return 1 / (1 + np.exp(-(a - b) / temperature))

def simulate_game(team_a, team_b, mode="prob", temperature=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    if mode == "chalk":
        return team_a if get_power(team_a) >= get_power(team_b) else team_b

    p_a = win_prob(team_a, team_b, temperature=temperature)
    return team_a if rng.random() < p_a else team_b

def play_round(matchups, round_name, mode="prob", temperature=0.5, rng=None):
    winners = []
    games = []

    for team_a, team_b in matchups:
        winner = simulate_game(team_a, team_b, mode=mode, temperature=temperature, rng=rng)
        loser = team_b if winner == team_a else team_a
        winners.append(winner)
        games.append({
            "round": round_name,
            "team_a": team_a,
            "team_b": team_b,
            "winner": winner,
            "loser": loser
        })

    return winners, games

def pair_round(teams):
    return [(teams[i], teams[i+1]) for i in range(0, len(teams), 2)]

def simulate_region(region_name, region_matchups, mode="prob", temperature=0.5, rng=None):
    r64_winners, r64_games = play_round(region_matchups, "R64", mode, temperature, rng)
    r32_winners, r32_games = play_round(pair_round(r64_winners), "R32", mode, temperature, rng)
    s16_winners, s16_games = play_round(pair_round(r32_winners), "S16", mode, temperature, rng)
    e8_winners, e8_games = play_round(pair_round(s16_winners), "E8", mode, temperature, rng)

    all_games = []
    for g in r64_games + r32_games + s16_games + e8_games:
        g["region"] = region_name
        all_games.append(g)

    return {
        "champion": e8_winners[0],
        "games": all_games
    }

def simulate_full_bracket(bracket, mode="prob", temperature=0.5, seed=42):
    rng = np.random.default_rng(seed)

    east = simulate_region("East", bracket["East"], mode, temperature, rng)
    west = simulate_region("West", bracket["West"], mode, temperature, rng)
    south = simulate_region("South", bracket["South"], mode, temperature, rng)
    midwest = simulate_region("Midwest", bracket["Midwest"], mode, temperature, rng)

    ff_matchups = [
        (east["champion"], west["champion"]),
        (south["champion"], midwest["champion"])
    ]

    ff_winners, ff_games = play_round(ff_matchups, "F4", mode, temperature, rng)
    title_winners, title_games = play_round([(ff_winners[0], ff_winners[1])], "2ND", mode, temperature, rng)

    all_games = (
        east["games"]
        + west["games"]
        + south["games"]
        + midwest["games"]
        + [{"region": "Final Four", **g} for g in ff_games]
        + [{"region": "Championship", **g} for g in title_games]
    )

    return {
        "champion": title_winners[0],
        "games": pd.DataFrame(all_games)
    }

In [39]:
sim = simulate_full_bracket(bracket_2024, mode="chalk")
pred_bracket = sim["games"]

print("Predicted Champion:", sim["champion"])
print(pred_bracket)

Predicted Champion: Connecticut
   round            team_a        team_b         winner             loser  \
0    R64       Connecticut       Stetson    Connecticut           Stetson   
1    R64  Florida Atlantic  Northwestern   Northwestern  Florida Atlantic   
2    R64     San Diego St.           UAB  San Diego St.               UAB   
3    R64            Auburn          Yale         Auburn              Yale   
4    R64               BYU      Duquesne            BYU          Duquesne   
..   ...               ...           ...            ...               ...   
58   S16         Creighton     Tennessee      Creighton         Tennessee   
59    E8            Purdue     Creighton         Purdue         Creighton   
60    F4       Connecticut       Arizona    Connecticut           Arizona   
61    F4           Houston        Purdue        Houston            Purdue   
62   2ND       Connecticut       Houston    Connecticut           Houston   

          region  
0           East  
1    

In [40]:
# use 2024 teams only
real_2024 = cbb[cbb["YEAR"] == 2024][["TEAM", "POSTSEASON"]].copy()

# treat R68 same as R64
real_2024["POSTSEASON"] = real_2024["POSTSEASON"].replace({"R68": "R64"})

# ordered rounds
round_ordered = ["R64", "R32", "S16", "E8", "F4", "2ND", "Champions"]
round_i = {r: i for i, r in enumerate(round_ordered)}

# map each team to how far they actually went
real_lookup = dict(zip(real_2024["TEAM"], real_2024["POSTSEASON"]))

def real_game_winner(team_a, team_b, round_name):
    a_round = real_lookup[team_a]
    b_round = real_lookup[team_b]

    a_idx = round_i[a_round]
    b_idx = round_i[b_round]
    game_idx = round_i[round_name]

    # winner should have advanced past this round
    if a_idx > game_idx and b_idx <= game_idx:
        return team_a
    elif b_idx > game_idx and a_idx <= game_idx:
        return team_b
    else:
        return None  # should not happen if bracket/data are correct

# copy predicted bracket
compare_df = pred_bracket.copy()

# get real winner for each predicted game
compare_df["real_winner"] = compare_df.apply(
    lambda row: real_game_winner(row["team_a"], row["team_b"], row["round"]),
    axis=1
)

# mark correct predictions
compare_df["correct"] = compare_df["winner"] == compare_df["real_winner"]

# overall accuracy
overall_acc = compare_df["correct"].mean()

print("Overall bracket accuracy:", overall_acc)
print()
print(compare_df[["round", "team_a", "team_b", "winner", "real_winner", "correct"]])

# accuracy by round
round_acc = compare_df.groupby("round")["correct"].mean().sort_index()
print()
print("Accuracy by round:")
print(round_acc)

Overall bracket accuracy: 0.6507936507936508

   round            team_a        team_b         winner    real_winner  \
0    R64       Connecticut       Stetson    Connecticut    Connecticut   
1    R64  Florida Atlantic  Northwestern   Northwestern   Northwestern   
2    R64     San Diego St.           UAB  San Diego St.  San Diego St.   
3    R64            Auburn          Yale         Auburn           Yale   
4    R64               BYU      Duquesne            BYU       Duquesne   
..   ...               ...           ...            ...            ...   
58   S16         Creighton     Tennessee      Creighton      Tennessee   
59    E8            Purdue     Creighton         Purdue         Purdue   
60    F4       Connecticut       Arizona    Connecticut    Connecticut   
61    F4           Houston        Purdue        Houston         Purdue   
62   2ND       Connecticut       Houston    Connecticut    Connecticut   

    correct  
0      True  
1      True  
2      True  
3     Fal

In [41]:
print(f"Overall bracket accuracy: {100 * overall_acc:.2f}%")
print((100 * round_acc).round(2))

Overall bracket accuracy: 65.08%
round
2ND    100.00
E8      50.00
F4      50.00
R32     75.00
R64     71.88
S16     25.00
Name: correct, dtype: float64


In [42]:
summary = compare_df.groupby("round")["correct"].agg(["sum", "count", "mean"])
summary["pct"] = 100 * summary["mean"]
print(summary)

       sum  count     mean      pct
round                              
2ND      1      1  1.00000  100.000
E8       2      4  0.50000   50.000
F4       1      2  0.50000   50.000
R32     12     16  0.75000   75.000
R64     23     32  0.71875   71.875
S16      2      8  0.25000   25.000
